# DAAI_N1.4 — Chuyển đổi Silver Data sang 15 bảng chuẩn hóa 3NF

Notebook này đọc các file **Silver** (đã làm sạch) và tái cấu trúc thành **15 bảng đúng theo `Luoc_do_quan_he_3NF.docx`**,
mỗi bảng xuất ra **1 file CSV riêng** trong thư mục `OUTPUT_DIR`.

**Trước khi chạy:**
1. Sửa `INPUT_DIR` bên dưới trỏ tới thư mục chứa các file Silver của nhóm.
2. Sửa dict `INPUT_FILES` nếu tên file thật của nhóm khác với tên mặc định đang để (mình đặt theo tên đã thảo luận: `customer_silver.csv`, `geography_silver.csv`...).
3. Chạy lần lượt từ trên xuống — mỗi bảng có 1 mục Markdown giải thích + 1 cell code xử lý + xuất file.

**2 điểm cần nhóm tự xác nhận với dữ liệu thật trước khi coi là bảng cuối cùng** (đã ghi trong tài liệu 3NF, notebook KHÔNG tự động tách):
- `GEOGRAPHY`: nếu `district → city → region` là 1-1 cố định thì nên tách thêm DISTRICT/CITY/REGION.
- `PRODUCT`: nếu mỗi `category` chỉ ứng với đúng 1 `segment` thì nên tách bảng CATEGORY riêng.

Notebook có 1 cell kiểm tra nhanh 2 điều này ở cuối (mục 16) để nhóm tự quyết định.

## Bước 0: Cấu hình đường dẫn & tiện ích chung

In [1]:
import pandas as pd
import numpy as np
import os
import json

# ==== SỬA 2 DÒNG NÀY CHO ĐÚNG MÁY CỦA BẠN ====
INPUT_DIR = "./silver_data"      # thư mục chứa các file *_silver.csv / *_silver.json
OUTPUT_DIR = "./warehouse_3nf"   # thư mục sẽ chứa 15 file kết quả
# ================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Ánh xạ: tên file silver đầu vào (sửa lại nếu nhóm bạn đặt tên khác)
INPUT_FILES = {
    "geography":      "geography_silver.csv",
    "customer":        "customer_silver.csv",
    "product":         "product_silver.csv",
    "promotion":       "promotion_silver.csv",
    "order":           "orders_enriched_silver.csv",
    "order_items":     "order_items_silver.csv",
    "shipment":        "shipments_silver.csv",   # bảng gộp shipment+shipper thô, sẽ tách ở Bước 9-10
    "payment":         "payments_silver.csv",
    "returns":         "returns_silver.csv",
    "reviews":         "reviews_silver.csv",
    "inventory":       "inventory_silver.csv",
    "web_traffic":     "web_traffic_silver.csv",
}

def load(key):
    """Đọc 1 file silver theo key trong INPUT_FILES, hỗ trợ cả .csv và .json."""
    fname = INPUT_FILES[key]
    path = os.path.join(INPUT_DIR, fname)
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Không tìm thấy '{path}'. Kiểm tra lại INPUT_DIR hoặc tên file trong INPUT_FILES['{key}']."
        )
    if path.endswith(".json"):
        return pd.read_json(path)
    return pd.read_csv(path)

def export(df, table_name):
    """Xuất 1 DataFrame thành 1 file CSV riêng, tên đúng theo tên bảng trong lược đồ 3NF."""
    out_path = os.path.join(OUTPUT_DIR, f"{table_name}.csv")
    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"Đã xuất {table_name}.csv — {df.shape[0]} dòng, {df.shape[1]} cột -> {out_path}")
    return df


## Bước 1/15: GEOGRAPHY *(zip)*
Giữ nguyên cấu trúc — không đổi so với bản Silver.

In [ ]:
df_geo_raw = load("geography")

geography = df_geo_raw[["zip", "city", "region", "district"]].drop_duplicates(subset="zip").reset_index(drop=True)
export(geography, "geography")
geography.head()

## Bước 2/15: CUSTOMER *(customer_id)*
Bỏ cột `city` (dư thừa vì đã suy ra được qua `zip → GEOGRAPHY.city`), chỉ giữ `zip` làm khóa ngoại.

In [ ]:
df_cust_raw = load("customer")

cust_cols = ["customer_id", "zip", "signup_date", "gender", "age_group", "acquisition_channel"]
# Chỉ giữ các cột có tồn tại thật trong file (phòng khi tên cột khác 1 chút)
cust_cols = [c for c in cust_cols if c in df_cust_raw.columns]
customer = df_cust_raw[cust_cols].drop_duplicates(subset="customer_id").reset_index(drop=True)
export(customer, "customer")
customer.head()

## Bước 3/15: SALES_EMPLOYEE *(sales_employee_id)*
⚠️ **Chưa có file dữ liệu gốc cho bảng này** (đã ghi trong Data Dictionary: `sales_employee_id` là FK nhưng
không có bảng định nghĩa nhân viên bán hàng). Cell dưới đây tạo bảng **tạm thời** bằng cách lấy toàn bộ
`sales_employee_id` duy nhất xuất hiện trong ORDER, cột `name` để trống (`NaN`) — **cần nhóm bổ sung dữ liệu
thật (HR data) rồi thay thế file này**, notebook chỉ tạo khung sườn để không bị lỗi FK khi tạo DDL.

In [ ]:
df_order_raw_for_emp = load("order")

emp_col = "sales_employee_id" if "sales_employee_id" in df_order_raw_for_emp.columns else None
if emp_col is None:
    raise KeyError("Không tìm thấy cột sales_employee_id trong file order — kiểm tra lại tên cột.")

sales_employee = (
    df_order_raw_for_emp[[emp_col]]
    .dropna()
    .drop_duplicates()
    .rename(columns={emp_col: "sales_employee_id"})
    .reset_index(drop=True)
)
sales_employee["name"] = np.nan  # placeholder — cần bổ sung dữ liệu thật
export(sales_employee, "sales_employee")
sales_employee.head()

## Bước 4/15: PRODUCT *(product_id)*
Không đổi cấu trúc so với Silver (đã merge `epd.json` + `products.csv` ở bước Silver trước đó).

In [ ]:
df_prod_raw = load("product")

prod_cols = ["product_id", "product_name", "category", "segment", "size", "color", "price", "cogs"]
prod_cols = [c for c in prod_cols if c in df_prod_raw.columns]
product = df_prod_raw[prod_cols].drop_duplicates(subset="product_id").reset_index(drop=True)
export(product, "product")
product.head()

## Bước 5/15: PROMOTION *(promo_id)*
Không đổi cấu trúc so với Silver (đã merge `eprom.json` + `promotions.csv`).

In [ ]:
df_promo_raw = load("promotion")

promo_cols = ["promo_id", "promo_name", "promo_type", "discount_value", "start_date", "end_date",
              "applicable_category", "promo_channel", "stackable_flag", "min_order_value"]
promo_cols = [c for c in promo_cols if c in df_promo_raw.columns]
promotion = df_promo_raw[promo_cols].drop_duplicates(subset="promo_id").reset_index(drop=True)
export(promotion, "promotion")
promotion.head()

## Bước 6/15: ORDER *(order_id)*
Bỏ `city`, `region`, `district` (dư thừa qua `zip`), bỏ `category`, `segment` (dư thừa qua ORDER_ITEMS → PRODUCT),
bỏ `payment_method` (chuyển hẳn về PAYMENT — không giữ trùng nữa).

In [ ]:
df_order_raw = load("order")

order_cols = ["order_id", "order_date", "customer_id", "zip", "order_status",
              "device_type", "order_source", "sales_employee_id"]
order_cols = [c for c in order_cols if c in df_order_raw.columns]
order = df_order_raw[order_cols].drop_duplicates(subset="order_id").reset_index(drop=True)
export(order, "order")
order.head()

## Bước 7-8/15: ORDER_ITEMS *(order_id, product_id)* + ORDER_ITEM_PROMOTION *(order_id, product_id, promo_id)*
`order_items_silver.csv` đang chứa `promo_id` và `promo_id_2` (nhóm lặp — vi phạm 1NF).
Cell dưới **tách 2 cột này thành các dòng riêng** trong bảng `ORDER_ITEM_PROMOTION`, để 1 dòng hàng có thể
áp 0, 1, 2 hoặc nhiều khuyến mãi mà không cần thêm cột.

In [ ]:
df_oi_raw = load("order_items")

# ---- 7. ORDER_ITEMS: chỉ giữ các cột thuộc về chính dòng hàng ----
oi_cols = ["order_id", "product_id", "quantity", "unit_price", "discount_amount"]
oi_cols = [c for c in oi_cols if c in df_oi_raw.columns]
order_items = df_oi_raw[oi_cols].drop_duplicates(subset=["order_id", "product_id"]).reset_index(drop=True)
export(order_items, "order_items")

# ---- 8. ORDER_ITEM_PROMOTION: unpivot promo_id + promo_id_2 thành nhiều dòng ----
promo_source_cols = [c for c in ["promo_id", "promo_id_2"] if c in df_oi_raw.columns]

order_item_promotion = (
    df_oi_raw[["order_id", "product_id"] + promo_source_cols]
    .melt(id_vars=["order_id", "product_id"], value_vars=promo_source_cols, value_name="promo_id")
    .drop(columns="variable")
    .dropna(subset=["promo_id"])
    .drop_duplicates(subset=["order_id", "product_id", "promo_id"])
    .sort_values(["order_id", "product_id"])
    .reset_index(drop=True)
)
export(order_item_promotion, "order_item_promotion")
order_items.head()

In [ ]:
order_item_promotion.head()

## Bước 9/15: SHIPPER *(shipper_id)*
File `shipments_silver.csv` gốc chứa cả thông tin shipper (tên, tuổi, xe...) lẫn `city/region/district`
nhưng **không có sẵn `zip`** — cần ánh xạ ngược qua bảng GEOGRAPHY (khớp theo `city+region+district`).
⚠️ Nếu tổ hợp `(city, region, district)` không map ra đúng **1** zip duy nhất (VD: 1 quận có nhiều mã zip),
cell dưới sẽ in cảnh báo — cần nhóm xác nhận lại dữ liệu thật, không tự ý chọn đại 1 zip.

In [ ]:
df_ship_raw = load("shipment")

shipper_cols_raw = ["shipper_id", "shipper_name", "shipper_phone", "shipper_gender", "shipper_age",
                     "shipper_marital_status", "shipper_education", "shipper_company", "shipper_vehicle",
                     "shipper_experience_years", "shipper_rating", "delivery_success_rate",
                     "average_delivery_time", "working_shift", "join_date", "city", "region", "district"]
shipper_cols_raw = [c for c in shipper_cols_raw if c in df_ship_raw.columns]

shipper_raw = df_ship_raw[shipper_cols_raw].drop_duplicates(subset="shipper_id").reset_index(drop=True)

# Map ngược (city, region, district) -> zip qua GEOGRAPHY
geo_lookup = geography[["zip", "city", "region", "district"]].drop_duplicates()
dup_check = geo_lookup.groupby(["city", "region", "district"])["zip"].nunique()
ambiguous = dup_check[dup_check > 1]
if len(ambiguous) > 0:
    print(f"CẢNH BÁO: {len(ambiguous)} tổ hợp (city, region, district) ứng với NHIỀU hơn 1 zip "
          f"— cần xác nhận thủ công, notebook sẽ lấy tạm zip đầu tiên tìm thấy.")

shipper = shipper_raw.merge(geo_lookup, on=["city", "region", "district"], how="left")
n_missing_zip = shipper["zip"].isna().sum()
if n_missing_zip > 0:
    print(f"CẢNH BÁO: {n_missing_zip} shipper không map được ra zip "
          f"(city/region/district không khớp GEOGRAPHY) — cần kiểm tra lại dữ liệu.")

shipper = shipper.drop(columns=["city", "region", "district"])
export(shipper, "shipper")
shipper.head()

## Bước 10/15: SHIPMENT *(order_id, shipper_id)*
Chỉ giữ các cột thuộc về **sự kiện giao hàng**, tách khỏi thông tin cá nhân shipper (đã tách ở Bước 9).

In [ ]:
shipment_cols = ["order_id", "shipper_id", "ship_date", "delivery_date", "shipping_fee"]
shipment_cols = [c for c in shipment_cols if c in df_ship_raw.columns]
shipment = df_ship_raw[shipment_cols].drop_duplicates(subset=["order_id", "shipper_id"]).reset_index(drop=True)
export(shipment, "shipment")
shipment.head()

## Bước 11/15: PAYMENT *(order_id)*
Không đổi cấu trúc so với Silver.

In [ ]:
df_pay_raw = load("payment")
pay_cols = ["order_id", "payment_method", "payment_value", "installments"]
pay_cols = [c for c in pay_cols if c in df_pay_raw.columns]
payment = df_pay_raw[pay_cols].drop_duplicates(subset="order_id").reset_index(drop=True)
export(payment, "payment")
payment.head()

## Bước 12/15: RETURNS *(return_id)*
Không đổi cấu trúc so với Silver — lưu ý FK `(order_id, product_id)` giờ tham chiếu tới composite key của ORDER_ITEMS.

In [ ]:
df_ret_raw = load("returns")
ret_cols = ["return_id", "order_id", "product_id", "return_date", "return_reason",
            "return_quantity", "refund_amount"]
ret_cols = [c for c in ret_cols if c in df_ret_raw.columns]
returns = df_ret_raw[ret_cols].drop_duplicates(subset="return_id").reset_index(drop=True)
export(returns, "returns")
returns.head()

## Bước 13/15: REVIEWS *(review_id)*
Không đổi cấu trúc so với Silver — giữ `customer_id` độc lập (người review không nhất thiết là người đặt đơn).

In [ ]:
df_rev_raw = load("reviews")
rev_cols = ["review_id", "order_id", "product_id", "customer_id", "review_date", "rating", "review_title"]
rev_cols = [c for c in rev_cols if c in df_rev_raw.columns]
reviews = df_rev_raw[rev_cols].drop_duplicates(subset="review_id").reset_index(drop=True)
export(reviews, "reviews")
reviews.head()

## Bước 14/15: INVENTORY *(snapshot_date, product_id)*
Bỏ `product_name`, `category`, `segment` (phụ thuộc bộ phận vào `product_id` — vi phạm 2NF),
bỏ `year`, `month` (phụ thuộc bộ phận vào `snapshot_date`, tính lại bằng hàm ngày khi cần).

In [ ]:
df_inv_raw = load("inventory")
inv_cols = ["snapshot_date", "product_id", "stock_on_hand", "units_received", "units_sold",
            "stockout_days", "days_of_supply", "fill_rate", "stockout_flag", "overstock_flag",
            "reorder_flag", "sell_through_rate"]
inv_cols = [c for c in inv_cols if c in df_inv_raw.columns]
inventory = df_inv_raw[inv_cols].drop_duplicates(subset=["snapshot_date", "product_id"]).reset_index(drop=True)
export(inventory, "inventory")
inventory.head()

## Bước 15/15: WEB_TRAFFIC *(date)*
Không đổi cấu trúc so với Silver.

In [ ]:
df_web_raw = load("web_traffic")
web_cols = ["date", "sessions", "unique_visitors", "page_views", "bounce_rate",
            "avg_session_duration_sec", "traffic_source"]
web_cols = [c for c in web_cols if c in df_web_raw.columns]
web_traffic = df_web_raw[web_cols].drop_duplicates(subset="date").reset_index(drop=True)
export(web_traffic, "web_traffic")
web_traffic.head()

## Bước 16: Kiểm tra nhanh 2 điểm còn "treo" trong tài liệu 3NF
Chạy 2 cell dưới để tự kiểm tra bằng dữ liệu thật xem có nên tách thêm bảng hay không — notebook KHÔNG tự tách,
chỉ in ra kết quả để nhóm quyết định.

In [ ]:
# Kiểm tra GEOGRAPHY: district -> city -> region có phải 1-1 không?
check_geo = geography.groupby("district")[["city", "region"]].nunique()
n_violation_geo = (check_geo[["city", "region"]] > 1).any(axis=1).sum()
print(f"Số district ứng với NHIỀU hơn 1 city/region: {n_violation_geo}")
print("-> Nếu = 0: có thể tách DISTRICT/CITY/REGION thành bảng riêng.")
print("-> Nếu > 0: GIỮ NGUYÊN 1 bảng GEOGRAPHY như hiện tại (không phải phụ thuộc bắc cầu thật).")

In [ ]:
# Kiểm tra PRODUCT: category -> segment có phải 1-1 không?
check_prod = product.groupby("category")["segment"].nunique()
n_violation_prod = (check_prod > 1).sum()
print(f"Số category ứng với NHIỀU hơn 1 segment: {n_violation_prod}")
print("-> Nếu = 0: có thể tách bảng CATEGORY(category_id, segment) riêng.")
print("-> Nếu > 0: GIỮ NGUYÊN PRODUCT như hiện tại.")

## Tổng kết: xác nhận đủ 15 file đã xuất
Liệt kê lại toàn bộ file trong `OUTPUT_DIR` kèm số dòng, đối chiếu với danh sách 15 bảng trong
`Luoc_do_quan_he_3NF.docx`.

In [ ]:
expected_tables = [
    "geography", "shipper", "customer", "sales_employee", "product", "promotion",
    "order", "order_items", "order_item_promotion", "shipment", "payment",
    "returns", "reviews", "inventory", "web_traffic",
]

print(f"{'Bảng':<25}{'Trạng thái':<15}{'Số dòng'}")
print("-" * 55)
for t in expected_tables:
    fp = os.path.join(OUTPUT_DIR, f"{t}.csv")
    if os.path.exists(fp):
        n = sum(1 for _ in open(fp, encoding="utf-8-sig")) - 1
        print(f"{t:<25}{'OK':<15}{n}")
    else:
        print(f"{t:<25}{'THIẾU':<15}-")